In [38]:
"""
HomeGuard Security System Simulator
Author: [Momina]
Description: A smart home monitoring system that processes sensor readings
             and triggers alerts for security, safety, and comfort issues.
"""
 
import random
from datetime import datetime
import time
 
# System configuration
HOME_MODES = ["HOME", "AWAY", "SLEEP"]
ALERT_SEVERITIES = ["LOW", "MEDIUM", "HIGH", "CRITICAL"]
 
# Current system state
current_mode = "AWAY"

In [39]:

def create_sensor(sensor_id, location, sensor_type, threshold=None):
    """
    Creates a sensor data structure.
    
    Parameters:
    - sensor_id: Unique identifier for the sensor
    - location: Where the sensor is located (e.g., "Living Room")
    - sensor_type: Type of sensor ("motion", "temperature", "door", "smoke")
    - threshold: Optional threshold value for the sensor
    
    Returns:
    - A dictionary representing the sensor
    """
    sensor = {"id": sensor_id,
    "location": location,
    "type": sensor_type,
    "threshold": threshold,
    "current_value": None
    }
    return sensor
 
def create_alert(severity, message, sensor_id, timestamp):
    """
    Creates an alert data structure.
    
    Parameters:
    - severity: Alert severity level (LOW, MEDIUM, HIGH, CRITICAL)
    - message: Description of the alert
    - sensor_id: ID of the sensor that triggered the alert
    - timestamp: When the alert was triggered
    
    Returns:
    - A dictionary representing the alert
    """

    alert = {"severity": severity,
             "message":message,
             "sensor_id":sensor_id,
             "timestamp":timestamp}
    return alert
 
# Initialize sensors for the Peterson home
sensors = [
    create_sensor("M001", "Living Room", "motion"), #motion for the living room 
    create_sensor("T001", "Kitchen", "temperature", threshold=75), #temperature for the kitchen with threshold
    create_sensor("D001", "Front Door", "door"), #door for the front door
    create_sensor("S001", "Bedroom", "smoke")  #smoke for the bedroom 
]
print(f"Initialized {len(sensors)} sensors")
for sensor in sensors:
    print(f"  - {sensor['id']}: {sensor['location']} ({sensor['type']})")


Initialized 4 sensors
  - M001: Living Room (motion)
  - T001: Kitchen (temperature)
  - D001: Front Door (door)
  - S001: Bedroom (smoke)


In [40]:
def is_abnormal_reading(sensor, reading_value):
    """
    Checks if a sensor reading is abnormal based on sensor type and thresholds.
    
    Parameters:
    - sensor: Sensor dictionary
    - reading_value: The current reading from the sensor
    
    Returns:
    - True if the reading is abnormal, False otherwise
    """
    sensor_type = sensor["type"]

    if sensor_type == "temperature":
        return reading_value < 35 or reading_value > 95
    elif sensor_type == "motion":
        return reading_value is True
    elif sensor_type == "door":
        return reading_value == "OPEN"
    elif sensor_type == "smoke":
        return reading_value == "DETECTED"

    return False
    
 
 
def should_trigger_security_alert(sensor, reading_value, system_mode):
    """
    Determines if a security alert should be triggered.
    
    Parameters:
    - sensor: Sensor dictionary
    - reading_value: The current reading from the sensor
    - system_mode: Current system mode (HOME, AWAY, SLEEP)
    
    Returns:
    - True if a security alert should be triggered, False otherwise
    """
    if system_mode == "AWAY" and sensor["type"] == "motion" and reading_value == True:
        return True
    elif system_mode == "AWAY" and sensor["type"] == "door" and reading_value == "OPEN":
        return True
    else: return False

# Test temperature check
test_sensor = create_sensor("TEMP_TEST", "Test Room", "temperature", threshold=35)
print(f"34°F is abnormal: {is_abnormal_reading(test_sensor, 34)}")  # Should be True
print(f"68°F is abnormal: {is_abnormal_reading(test_sensor, 68)}")  # Should be False
 
# Test security alert
motion_sensor = create_sensor("MOTION_TEST", "Test Room", "motion")
print(f"Motion in AWAY mode triggers alert: {should_trigger_security_alert(motion_sensor, True, 'AWAY')}")  # Should be True
print(f"Motion in HOME mode triggers alert: {should_trigger_security_alert(motion_sensor, True, 'HOME')}")  # Should be False
    

34°F is abnormal: True
68°F is abnormal: False
Motion in AWAY mode triggers alert: True
Motion in HOME mode triggers alert: False


In [41]:
def generate_reading(sensor):
    sensor_type = sensor["type"] if isinstance(sensor, dict) else sensor.sensor_type

    if sensor_type == "temperature":
        return random.randint(30, 100)
    elif sensor_type== "motion":
        return random.choice([True, False])     
    elif sensor_type == "door":
        return random.choice(["OPEN", "CLOSED"])
    elif sensor_type == "smoke":
        return random.choice(["DETECTED", "CLEAR"])

def process_reading(sensor, reading_value, system_mode):
    """
    Processes a sensor reading and determines if alerts are needed.
    
    Parameters:
    - sensor: Sensor dictionary
    - reading_value: The reading from the sensor
    - system_mode: Current system mode
    
    Returns:
    - A list of alerts (empty if no alerts needed)
    """
    alerts = []
    timestamp = datetime.now().isoformat()

# Security alerts
    if system_mode == "AWAY":
        if sensor["type"] == "motion" and reading_value == True:
            alerts.append(create_alert("HIGH", "Motion detected while away!", sensor["id"], timestamp))
        elif sensor["type"] == "door" and reading_value == "OPEN":
            alerts.append(create_alert("HIGH", "Door opened while away!", sensor["id"], timestamp))

     # Safety alerts
    if sensor["type"] == "temperature" and (reading_value < 35 or reading_value > 95):
        alerts.append(create_alert("MEDIUM", f"Abnormal temperature reading: {reading_value}°F", sensor["id"], timestamp))
    elif sensor["type"] == "smoke" and reading_value == "DETECTED":
        alerts.append(create_alert("HIGH", "Smoke detected!", sensor["id"], timestamp))

    
    # Comfort notifications (only in HOME mode) 
    # In HOME mode, added a LOW comfort alert when temperature is below 65°F or above 75°F.
    if system_mode == "HOME":
            if sensor["type"] == "temperature" and (reading_value < 65 or reading_value > 75):
                alerts.append(create_alert(
    "LOW",
    f"Temperature outside comfort range: {reading_value}°F",
    sensor["id"],
    timestamp
))
                
    return alerts

# Feel free to change the function signatures or add helper functions as needed, these are just some suggestions to make the code more friendly! 
def trigger_alert(alert):
    """
    Displays an alert to the user.
    
    Parameters:
    - alert: Alert dictionary
    """
    severity_symbol = {
        "LOW": "ℹ️",
        "MEDIUM": "⚠️",
        "HIGH": "🚨",
        "CRITICAL": "🔥"
    }
    
    symbol = severity_symbol.get(alert["severity"], "⚠️")
    print(f"[ALERT!] {symbol} {alert['severity']}: {alert['message']}")

    
def log_event(message, timestamp=None):
    """
    Logs an event to the console.
    
    Parameters:
    - message: The message to log
    - timestamp: Optional timestamp (uses current time if not provided)
    """
    if timestamp is None:
        timestamp = datetime.now().strftime("%H:%M:%S")
    print(f"[LOG] [{timestamp}] {message}")



# Test reading generation
test_sensor = sensors[0]  # Motion sensor
reading = generate_reading(test_sensor)
print(f"Generated reading for {test_sensor['location']}: {reading}")
 
# Test processing
alerts = process_reading(test_sensor, True, "AWAY")
if alerts:
    trigger_alert(alerts[0])

Generated reading for Living Room: True
[ALERT!] 🚨 HIGH: Motion detected while away!


In [42]:
#part 5 we’re making a sensor that can remember its own details
class Sensor:
    """
    Represents a sensor in the HomeGuard system.
    """
    
    def __init__(self, sensor_id, location, sensor_type, threshold=None):
        self.sensor_id = sensor_id
        self.location = location
        self.sensor_type = sensor_type
        self.threshold = threshold
        self.current_value = None
        """
        Initializes a new sensor.
        
        Parameters:
        - sensor_id: Unique identifier for the sensor
        - location: Where the sensor is located
        - sensor_type: Type of sensor ("motion", "temperature", "door", "smoke")
        - threshold: Optional threshold value for the sensor
        """


        
        
        """
        Generates and stores a new reading for this sensor.
        
        Returns:
        - The reading value
        """
          # YOUR CODE HERE: Generate a reading and store it in self.current_value
        
    def read(self):
            
        sensor_reading = generate_reading(self) #changed after suggestion to fix
        self.current_value = sensor_reading
        return self.current_value
        
        
    """
        Checks if the current reading is abnormal. 
        #This method’s job is to check a reading the sensor already has and tell us whether it is abnormal.
        
        Returns:
        - True if the reading is abnormal, False otherwise
        """
        # YOUR CODE HERE: Use the is_abnormal_reading function logic

    def isAbnormal(self):

           if self.current_value is None: #added after suggestion to fix
                     return False
           if self.sensor_type == "temperature":
                if self.current_value < 35 or self.current_value > 95:
                    return True
                else: 
                    return False
           
           elif self.sensor_type== "motion":
                if self.current_value == True:
                    return True
                else:
                    return False

           elif self.sensor_type == "door":
                if self.current_value == "OPEN":
                    return True
                else:
                    return False

           elif self.sensor_type == "smoke":
                if self.current_value == "DETECTED":
                    return True
                else:
                    return False    

    def __str__(self):
        """
        Returns a string representation of the sensor.
        """
        if self.current_value is None:
            status = "No reading"
        else:
            status = str(self.current_value)

        return f"{self.sensor_id} ({self.location}): {status}"     

    # Create sensor objects using the class
sensor_objects = [
    Sensor("MOTION_001", "Living Room", "motion"),
    Sensor("TEMP_001", "Kitchen", "temperature", threshold=35),
    Sensor("DOOR_001", "Front Door", "door"),
    Sensor("SMOKE_001", "Bedroom", "smoke")
]

# Create and test a sensor
test_sensor = Sensor("TEST_001", "Test Room", "temperature", threshold=35)
test_sensor.read()
print(f"Sensor reading: {test_sensor.current_value}")
print(f"Is abnormal: {test_sensor.isAbnormal()}")
print(f"Sensor info: {test_sensor}")
   

                

Sensor reading: 98
Is abnormal: True
Sensor info: TEST_001 (Test Room): 98


In [43]:
def run_simulation(duration_minutes=5, system_mode="AWAY"):
    """
    Runs the HomeGuard security system simulation.
    
    Parameters:
    - duration_minutes: How long to run the simulation (default: 5 minutes)
    - system_mode: System mode (HOME, AWAY, SLEEP)
    """

    print("=" * 50)
    print("=== HomeGuard Security System ===")
    print("=" * 50)
    print(f"Mode: {system_mode}")

    sensors = [
        Sensor("MOTION_001", "Living Room", "motion"),
        Sensor("TEMP_001", "Kitchen", "temperature", threshold=35),
        Sensor("DOOR_001", "Front Door", "door"),
        Sensor("SMOKE_001", "Bedroom", "smoke")
    ]

    for minute in range(duration_minutes):
        current_time = datetime.now().strftime("%H:%M:%S")
        print(f"\nTime: {current_time}")  

     # Read all sensors
        for sensor in sensors:
            reading = sensor.read()

    # Display the reading
            if sensor.sensor_type == "temperature":
                status = "Normal" if 65 <= reading <= 75 else "Abnormal"
                print(f"[READING] {sensor.location} Temperature: {reading}°F ({status})")
            elif sensor.sensor_type == "motion":
                status = "DETECTED" if reading else "No activity"
                print(f"[READING] {sensor.location} Motion: {status}")
            elif sensor.sensor_type == "door":
                print(f"[READING] {sensor.location}: {reading}")
            elif sensor.sensor_type == "smoke":
                print(f"[READING] {sensor.location} Smoke: {reading}")

     # Process the reading and trigger alerts if needed
            # Convert sensor object to dict format for process_reading function
            sensor_dict = {
                "id": sensor.sensor_id,
                "location": sensor.location,
                "type": sensor.sensor_type,
                "threshold": sensor.threshold
            }
            alerts = process_reading(sensor_dict, reading, system_mode)

    # Trigger all alerts
            for alert in alerts:
                trigger_alert(alert)
                if alert["severity"] in ["HIGH", "CRITICAL"]:
                    log_event("Sending notification to homeowner...")

       
        # Small delay to make output readable (optional)
        import time
        time.sleep(0.5)  # 0.5 second delay between iterations

    print("\n" + "=" * 50)
    print("Simulation complete!")
    print("=" * 50)

# Main execution simulation for 3 minutes in mode AWAY
if __name__ == "__main__":
    # Run the simulation
    run_simulation(duration_minutes=3, system_mode="AWAY")

        
                


=== HomeGuard Security System ===
Mode: AWAY

Time: 19:27:21
[READING] Living Room Motion: DETECTED
[ALERT!] 🚨 HIGH: Motion detected while away!
[LOG] [19:27:21] Sending notification to homeowner...
[READING] Kitchen Temperature: 77°F (Abnormal)
[READING] Front Door: CLOSED
[READING] Bedroom Smoke: CLEAR

Time: 19:27:22
[READING] Living Room Motion: DETECTED
[ALERT!] 🚨 HIGH: Motion detected while away!
[LOG] [19:27:22] Sending notification to homeowner...
[READING] Kitchen Temperature: 76°F (Abnormal)
[READING] Front Door: OPEN
[ALERT!] 🚨 HIGH: Door opened while away!
[LOG] [19:27:22] Sending notification to homeowner...
[READING] Bedroom Smoke: DETECTED
[ALERT!] 🚨 HIGH: Smoke detected!
[LOG] [19:27:22] Sending notification to homeowner...

Time: 19:27:22
[READING] Living Room Motion: DETECTED
[ALERT!] 🚨 HIGH: Motion detected while away!
[LOG] [19:27:22] Sending notification to homeowner...
[READING] Kitchen Temperature: 77°F (Abnormal)
[READING] Front Door: CLOSED
[READING] Bedroom Sm

In [44]:
#The 30 reading is below safety limit of 35°F

test_alerts = process_reading(sensors[1], 30, "AWAY")
for alert in test_alerts:
    trigger_alert(alert)

[ALERT!] ⚠️ MEDIUM: Abnormal temperature reading: 30°F


In [45]:
#SAFETY TEST: TEMPERATURE TEST: Test status=home, sensor temperature

test_alerts = process_reading(sensors[1], 30, "HOME")
for alert in test_alerts:
    trigger_alert(alert)

[ALERT!] ⚠️ MEDIUM: Abnormal temperature reading: 30°F
[ALERT!] ℹ️ LOW: Temperature outside comfort range: 30°F


In [46]:
#SECURITY TEST: DOOR OPEN
test_alerts = process_reading(sensors[2], "OPEN", "AWAY")
for alert in test_alerts:
    trigger_alert(alert)

[ALERT!] 🚨 HIGH: Door opened while away!


In [47]:
#SAFETY TEST: Smoke alert

test_alerts = process_reading(sensors[3], "DETECTED", "AWAY")
for alert in test_alerts:
    trigger_alert(alert)

[ALERT!] 🚨 HIGH: Smoke detected!


In [48]:
#COMFORT TEST: Running comfort test with temperature reading of 60°F in HOME mode, which is below the comfort range of 65-75°F
comfort_test_alerts = process_reading(sensors[1], 60, "HOME")

for alert in comfort_test_alerts:
    trigger_alert(alert)

[ALERT!] ℹ️ LOW: Temperature outside comfort range: 60°F


In [49]:
#Edge Test: temperature reading of 65F which is on the edge of the comfort range, to see if it creates a comfort alert or not. 

comfort_test_alerts = process_reading(sensors[1], 65, "HOME")

for alert in comfort_test_alerts:
    trigger_alert(alert)
print(comfort_test_alerts)

[]


In [50]:
#simulation to check home mode with motion sensor
run_simulation(duration_minutes=3, system_mode="HOME")

=== HomeGuard Security System ===
Mode: HOME

Time: 19:27:23
[READING] Living Room Motion: DETECTED
[READING] Kitchen Temperature: 36°F (Abnormal)
[ALERT!] ℹ️ LOW: Temperature outside comfort range: 36°F
[READING] Front Door: CLOSED
[READING] Bedroom Smoke: CLEAR

Time: 19:27:23
[READING] Living Room Motion: No activity
[READING] Kitchen Temperature: 90°F (Abnormal)
[ALERT!] ℹ️ LOW: Temperature outside comfort range: 90°F
[READING] Front Door: OPEN
[READING] Bedroom Smoke: DETECTED
[ALERT!] 🚨 HIGH: Smoke detected!
[LOG] [19:27:23] Sending notification to homeowner...

Time: 19:27:24
[READING] Living Room Motion: DETECTED
[READING] Kitchen Temperature: 53°F (Abnormal)
[ALERT!] ℹ️ LOW: Temperature outside comfort range: 53°F
[READING] Front Door: OPEN
[READING] Bedroom Smoke: CLEAR

Simulation complete!


In [51]:
#simluation to check away mode with motion sensor
run_simulation(duration_minutes=3, system_mode="SLEEP")

=== HomeGuard Security System ===
Mode: SLEEP

Time: 19:27:24
[READING] Living Room Motion: No activity
[READING] Kitchen Temperature: 64°F (Abnormal)
[READING] Front Door: CLOSED
[READING] Bedroom Smoke: DETECTED
[ALERT!] 🚨 HIGH: Smoke detected!
[LOG] [19:27:24] Sending notification to homeowner...

Time: 19:27:25
[READING] Living Room Motion: No activity
[READING] Kitchen Temperature: 97°F (Abnormal)
[ALERT!] ⚠️ MEDIUM: Abnormal temperature reading: 97°F
[READING] Front Door: OPEN
[READING] Bedroom Smoke: CLEAR

Time: 19:27:25
[READING] Living Room Motion: No activity
[READING] Kitchen Temperature: 74°F (Normal)
[READING] Front Door: OPEN
[READING] Bedroom Smoke: DETECTED
[ALERT!] 🚨 HIGH: Smoke detected!
[LOG] [19:27:25] Sending notification to homeowner...

Simulation complete!


In [52]:
# Run three rounds in AWAY mode
run_simulation(duration_minutes=3, system_mode="AWAY")

=== HomeGuard Security System ===
Mode: AWAY

Time: 19:27:26
[READING] Living Room Motion: DETECTED
[ALERT!] 🚨 HIGH: Motion detected while away!
[LOG] [19:27:26] Sending notification to homeowner...
[READING] Kitchen Temperature: 40°F (Abnormal)
[READING] Front Door: CLOSED
[READING] Bedroom Smoke: DETECTED
[ALERT!] 🚨 HIGH: Smoke detected!
[LOG] [19:27:26] Sending notification to homeowner...

Time: 19:27:26
[READING] Living Room Motion: DETECTED
[ALERT!] 🚨 HIGH: Motion detected while away!
[LOG] [19:27:26] Sending notification to homeowner...
[READING] Kitchen Temperature: 41°F (Abnormal)
[READING] Front Door: CLOSED
[READING] Bedroom Smoke: CLEAR

Time: 19:27:27
[READING] Living Room Motion: No activity
[READING] Kitchen Temperature: 83°F (Abnormal)
[READING] Front Door: OPEN
[ALERT!] 🚨 HIGH: Door opened while away!
[LOG] [19:27:27] Sending notification to homeowner...
[READING] Bedroom Smoke: CLEAR

Simulation complete!
